# Part 5 · 00 Setup and input check

This notebook runs once before anything else in part 5. It:

1. records the environment (Python and package versions, for the reproducibility checklist);
2. checks that every input from parts 1–2 and 3–4 is where part 5 expects it, and says which mode `cache_raw.py` will run in;
3. runs the unit tests of the cleaning on four kinds of document (no data, no network).

Part 5 needs neither WRDS nor any login: the WRDS data it uses (CRSP names, I/B/E/S link) were already pulled by part 3–4.

**Before running:** upload the files of `code/` to `S08 NLP/5_data cleaning/code/`.

**Folder layout**

```
S08 NLP/5_data cleaning/
  code/     settings.py (paths), cache_raw, clean_release (+ tests), entities,
            token_stats, dedup, model_cache, splits, manifest
  cache/    raw EDGAR bytes (raw/raw_YYYY.zip + raw_manifest.parquet) and model outputs
  data/     built tables keyed by accession: clean/, chunks/ (pieces per model), entities,
            token_stats, dedup_flags, splits
  results/  tables for the report
  logs/     environment records
```

**Inputs read, never changed:** `1_2nov/nov2_data` (event table, extraction report, timing), `1_2nov/files .py/fetch_releases.py`, and from `3_4_surprise_returns` the release clock, the trading calendar, `crsp_names.parquet`, `link_ibes_crsp.parquet` and `SPLITS` in `common.py`.

In [ ]:
!pip install -q beautifulsoup4 lxml nltk pyarrow datasketch tiktoken transformers
# If Colab asks you to restart the session after this install, restart it and
# continue from the next cell.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, sys
from pathlib import Path

# The shared team folder. Leave TEAM_DIR = None to look for "S08 NLP" in My Drive
# (also one or two folders down), in the shared drives and among the shortcuts;
# or give its path, e.g. Path("/content/drive/MyDrive/MIT/S08 NLP").
TEAM_DIR = None


def find_team_dir(name="S08 NLP"):
    roots = [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives"),
             Path("/content/drive/.shortcut-targets-by-id")]
    for pattern in (name, f"*/{name}", f"*/*/{name}"):
        for root in roots:
            hits = sorted(p for p in root.glob(pattern) if (p / "1_2nov").is_dir())
            if hits:
                return hits[0]
    return None


TEAM_DIR = Path(TEAM_DIR) if TEAM_DIR else find_team_dir()
if TEAM_DIR is None:
    top = sorted(p.name for p in Path("/content/drive/MyDrive").iterdir())
    raise FileNotFoundError(
        "No 'S08 NLP' folder (with 1_2nov inside) in this Drive. Either add a shortcut "
        "to the shared folder in My Drive (right-click it in 'Shared with me', "
        "Organise, Add shortcut, My Drive) and rerun, or set TEAM_DIR above to its "
        f"path. Top of My Drive: {top}")
STEP5_DIR = TEAM_DIR / "5_data cleaning"
CODE = STEP5_DIR / "code"
assert STEP5_DIR.exists(), f"{STEP5_DIR} not found: upload the '5_data cleaning' folder there"

for sub in ["code", "cache", "data", "results", "logs"]:
    (STEP5_DIR / sub).mkdir(parents=True, exist_ok=True)

os.environ["TEAM_DIR"] = str(TEAM_DIR)
os.environ["STEP5_DIR"] = str(STEP5_DIR)
sys.path.insert(0, str(CODE))
os.chdir(STEP5_DIR)       # the scripts are run as python code/<script>.py

missing = [f for f in ["settings.py", "cache_raw.py", "clean_release.py", "test_clean_release.py",
                                 "entities.py", "token_stats.py", "dedup.py", "model_cache.py",
                                 "splits.py", "manifest.py"] if not (CODE / f).exists()]
assert not missing, f"upload these to {CODE}: {missing}"
print("step 5 folder:", STEP5_DIR)
print("code files:   ", sorted(os.listdir(CODE)))

In [ ]:
# Environment record: goes into logs/ and, later, the README.
import json, platform, datetime as dt
import importlib.metadata as md

env = {"run_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
       "python": platform.python_version()}
for pkg in ["pandas", "numpy", "pyarrow", "beautifulsoup4", "lxml", "nltk",
            "datasketch", "tiktoken", "transformers"]:
    try:
        env[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        env[pkg] = "not installed"
(STEP5_DIR / "logs" / f"env_{dt.date.today()}.json").write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))

## Inputs from the other parts

Every file part 5 reads, with its size. `extraction_report_full.csv` is the full report of `fetch_releases.py` (with `accession`, `cik` and `filename`); the shared `extraction_report.csv` has only `source`, `words` and `tables_dropped`. With the full report, `cache_raw.py` makes one request per document (3 to 4 hours); without it, it reads each index page again (7 to 8 hours).

In [ ]:
import pandas as pd
import settings as S

inputs = {
    "event table (1_2nov)": S.EVENT_TABLE,
    "extraction report, shared (1_2nov)": S.EXTRACTION_REPORT,
    "three-source timing (1_2nov)": S.TIMING,
    "fetch_releases.py (1_2nov)": S.NOV2_CODE / "fetch_releases.py",
    "release clock (3_4)": S.CLOCK,
    "trading calendar (3_4)": S.CALENDAR,
    "CRSP names (3_4 cache)": S.CRSP_NAMES,
    "I/B/E/S-CRSP link (3_4 cache)": S.IBES_LINK,
    "common.py with SPLITS (3_4)": S.PART34 / "code" / "common.py",
}
for label, p in inputs.items():
    print(f"  {'ok ' if p.exists() else 'MISSING'}  {label:<38} "
          f"{p.stat().st_size if p.exists() else 0:>12,}")

full = S.extraction_full()
print()
print("cache_raw.py mode:", f"report ({full})" if full else
      "index (no extraction_report_full.csv: each index page is read again)")

clk = pd.read_csv(S.CLOCK, usecols=["accession", "reaction_date", "status"], dtype=str)
print(f"\nrelease clock: {len(clk):,} rows, {clk.accession.nunique():,} accessions, "
      f"{clk.reaction_date.notna().sum():,} with a reaction date")
print(clk.status.value_counts().to_string())
print("\nsplits from common.py:", S.common_3_4().SPLITS)

## Unit tests of the cleaning

Four small documents built like the four kinds met in the sample (a recent HTML release, an old release laid out in a table, an old `.txt` filing in SGML, an iXBRL 8-K body). Every line should start with `ok`.

In [ ]:
!python code/test_clean_release.py